# ZeroPoint Tetris Decision Bench

**CPU-first comparison** of decision models on the same Tetris shortlist task:

| Model | Role | Runs where |
|-------|------|------------|
| **ZeroPoint** | Local energy-based decision model | CPU (in-process) |
| **Laya** | Open System-One decision model (zero-shot) | **CPU** (forced) |
| **OpenAI Decisions** (`gpt-6-luna`) | Hosted decision API — **optional; not part of the published results** | Cloud |
| **Greedy shortlist** | Always pick rank-1 heuristic option | Baseline (no model) |
| **Random shortlist** | Uniform pick among shortlist | Floor |

> **Published experiment (see `results/`)**: greedy, random, ZeroPoint, Laya on CPU. **OpenAI Decisions was not run** — leave `run_openai = False` unless you are adding that arm yourself.

## What this measures (and what it does not)

**Measures:** given an identical shortlist of placements described as short text options, how well each *decision model* chooses, plus latency / cost / footprint.

**Does not measure:** raw Tetris skill from pixels. The Python engine enumerates legal landings, scores them with a fixed heuristic, and keeps the top-*K* options. Models only choose among that shortlist. Strong play therefore depends on the shortlist — that is intentional and documented.

## Fairness rules

1. Same board, piece, seed, shortlist, and option wording for every model.
2. Laya is forced onto **CPU** (`CUDA_VISIBLE_DEVICES=""`) so it matches ZeroPoint’s deployment class.
3. Zero-shot only — no Tetris fine-tunes.
4. OpenAI is optional and **off by default** — the published `results/` in this repo do **not** include OpenAI (that experiment was not run).

## How to run in Colab

1. Runtime → **CPU** (recommended for a fair ZeroPoint vs Laya race).
2. Set secret `OPENAI_API_KEY` if you want the OpenAI arm (🔑 icon / Secrets).
3. Runtime → Run all.

Expected wall time (CPU, 10 seeds × 100 moves, 3 models): roughly **tens of minutes to a few hours**, dominated by Laya on CPU and OpenAI RTT. Start with the smoke test cell.


## 0 — Force CPU and record the environment


In [ ]:
import os
import platform
import sys
import time

# Fair local comparison: never let Laya/torch grab a GPU in this notebook.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("python", sys.version.split()[0])
print("platform", platform.platform())
print("machine", platform.machine())
print("CUDA_VISIBLE_DEVICES", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
print("started", time.strftime("%Y-%m-%d %H:%M:%S"))


## 1 — Install dependencies


In [ ]:
%pip install -q "zeropoint==1.1.0" "laya>=0.5.0" "openai>=1.60.0" "pandas>=2.0" "matplotlib>=3.7" "tqdm>=4.66" "requests>=2.31"


## 2 — Experiment configuration

Edit this cell, then re-run from here.


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass
class Config:
    # Game
    seeds: tuple = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
    max_moves: int = 100          # pieces per game (cap); game may end earlier on top-out
    shortlist_k: int = 8          # options shown to every model
    board_width: int = 10
    board_height: int = 20

    # Which arms to run
    run_zeropoint: bool = True
    run_laya: bool = True
    run_openai: bool = False      # enable later when you have a working key
    run_greedy: bool = True       # heuristic rank-1 (shortlist ceiling)
    run_random: bool = True       # shortlist floor

    # OpenAI
    openai_model: str = "gpt-6-luna"
    openai_timeout_s: float = 60.0

    # Laya
    laya_model: str = "english"   # Router checkpoint name; CPU-only in this notebook

    # ZeroPoint
    zeropoint_threads: int = 2

    # Smoke test (fast path before the full run)
    smoke_seeds: tuple = (1,)
    smoke_max_moves: int = 15

    # Output
    results_dir: str = "results"

CFG = Config()
Path(CFG.results_dir).mkdir(parents=True, exist_ok=True)
print(asdict(CFG))


## 3 — OpenAI API key (optional)

In Colab: **Secrets** → add `OPENAI_API_KEY`. Locally: export the same env var or paste into the fallback below (not recommended).


In [ ]:
OPENAI_API_KEY = None

try:
    from google.colab import userdata  # type: ignore
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    print("OpenAI key: loaded from Colab Secrets")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
    if OPENAI_API_KEY:
        print("OpenAI key: loaded from environment")
    else:
        print("OpenAI key: missing — OpenAI arm will be skipped")

if not OPENAI_API_KEY:
    CFG.run_openai = False


## 4 — Shared Tetris engine

Standard 10×20 board, 7-bag randomizer, drop placements only (no soft-drop finesse / T-spins). The heuristic used for shortlisting is fixed and identical for every model.


In [ ]:
from __future__ import annotations

import copy
import math
import random
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

import numpy as np

# --- shapes: each rotation is a list of (x, y) offsets ---
SHAPES: dict[str, list[list[tuple[int, int]]]] = {
    "I": [
        [(0, 1), (1, 1), (2, 1), (3, 1)],
        [(2, 0), (2, 1), (2, 2), (2, 3)],
        [(0, 2), (1, 2), (2, 2), (3, 2)],
        [(1, 0), (1, 1), (1, 2), (1, 3)],
    ],
    "O": [
        [(1, 0), (2, 0), (1, 1), (2, 1)],
        [(1, 0), (2, 0), (1, 1), (2, 1)],
        [(1, 0), (2, 0), (1, 1), (2, 1)],
        [(1, 0), (2, 0), (1, 1), (2, 1)],
    ],
    "T": [
        [(1, 0), (0, 1), (1, 1), (2, 1)],
        [(1, 0), (1, 1), (2, 1), (1, 2)],
        [(0, 1), (1, 1), (2, 1), (1, 2)],
        [(1, 0), (0, 1), (1, 1), (1, 2)],
    ],
    "S": [
        [(1, 0), (2, 0), (0, 1), (1, 1)],
        [(1, 0), (1, 1), (2, 1), (2, 2)],
        [(1, 1), (2, 1), (0, 2), (1, 2)],
        [(0, 0), (0, 1), (1, 1), (1, 2)],
    ],
    "Z": [
        [(0, 0), (1, 0), (1, 1), (2, 1)],
        [(2, 0), (1, 1), (2, 1), (1, 2)],
        [(0, 1), (1, 1), (1, 2), (2, 2)],
        [(1, 0), (0, 1), (1, 1), (0, 2)],
    ],
    "J": [
        [(0, 0), (0, 1), (1, 1), (2, 1)],
        [(1, 0), (2, 0), (1, 1), (1, 2)],
        [(0, 1), (1, 1), (2, 1), (2, 2)],
        [(1, 0), (1, 1), (0, 2), (1, 2)],
    ],
    "L": [
        [(2, 0), (0, 1), (1, 1), (2, 1)],
        [(1, 0), (1, 1), (1, 2), (2, 2)],
        [(0, 1), (1, 1), (2, 1), (0, 2)],
        [(0, 0), (1, 0), (1, 1), (1, 2)],
    ],
}
PIECE_NAMES = list(SHAPES.keys())


def _holes(grid: np.ndarray) -> int:
    h, w = grid.shape
    holes = 0
    for x in range(w):
        block_seen = False
        for y in range(h):
            if grid[y, x]:
                block_seen = True
            elif block_seen:
                holes += 1
    return holes


def _column_heights(grid: np.ndarray) -> list[int]:
    h, w = grid.shape
    heights = []
    for x in range(w):
        col = grid[:, x]
        filled = np.where(col)[0]
        heights.append(0 if filled.size == 0 else int(h - filled[0]))
    return heights


def _bumpiness(heights: list[int]) -> int:
    return int(sum(abs(heights[i] - heights[i + 1]) for i in range(len(heights) - 1)))


def _aggregate_height(heights: list[int]) -> int:
    return int(sum(heights))


def board_features(grid: np.ndarray) -> dict[str, int]:
    heights = _column_heights(grid)
    return {
        "holes": _holes(grid),
        "bumpiness": _bumpiness(heights),
        "height": max(heights) if heights else 0,
        "aggregate_height": _aggregate_height(heights),
    }


@dataclass(frozen=True)
class Placement:
    piece: str
    rotation: int
    x: int
    y: int
    cells: tuple[tuple[int, int], ...]
    lines_cleared: int
    holes: int
    bumpiness: int
    height: int
    aggregate_height: int
    heuristic: float
    label: str = ""

    def option_text(self) -> str:
        return (
            f"clears {self.lines_cleared}, holes {self.holes}, "
            f"bumpiness {self.bumpiness}, height {self.height}"
        )


@dataclass
class GameState:
    grid: np.ndarray
    bag: list[str] = field(default_factory=list)
    queue: list[str] = field(default_factory=list)
    current: str = "I"
    lines: int = 0
    pieces: int = 0
    alive: bool = True
    rng: random.Random = field(default_factory=random.Random)

    def copy(self) -> "GameState":
        return GameState(
            grid=self.grid.copy(),
            bag=list(self.bag),
            queue=list(self.queue),
            current=self.current,
            lines=self.lines,
            pieces=self.pieces,
            alive=self.alive,
            rng=random.Random(self.rng.random()),  # independent stream
        )


class TetrisEngine:
    def __init__(self, width: int = 10, height: int = 20, seed: int = 1):
        self.width = width
        self.height = height
        self.state = GameState(grid=np.zeros((height, width), dtype=np.uint8), rng=random.Random(seed))
        self._refill_queue()
        self.state.current = self._pop_next()
        if not self._spawn_ok(self.state.current):
            self.state.alive = False

    def _refill_bag(self) -> None:
        bag = list(PIECE_NAMES)
        self.state.rng.shuffle(bag)
        self.state.bag.extend(bag)

    def _refill_queue(self) -> None:
        while len(self.state.queue) < 5:
            if not self.state.bag:
                self._refill_bag()
            self.state.queue.append(self.state.bag.pop(0))

    def _pop_next(self) -> str:
        self._refill_queue()
        return self.state.queue.pop(0)

    def _cells(self, piece: str, rotation: int, x: int, y: int) -> list[tuple[int, int]]:
        return [(x + dx, y + dy) for dx, dy in SHAPES[piece][rotation % 4]]

    def _valid(self, grid: np.ndarray, cells: list[tuple[int, int]]) -> bool:
        h, w = grid.shape
        for x, y in cells:
            if x < 0 or x >= w or y < 0 or y >= h:
                return False
            if grid[y, x]:
                return False
        return True

    def _spawn_ok(self, piece: str) -> bool:
        # Spawn near top-center; if the default pose collides, game over.
        for rot in range(4):
            cells = self._cells(piece, rot, 3, 0)
            if self._valid(self.state.grid, cells):
                return True
        return False

    def _drop_y(self, grid: np.ndarray, piece: str, rotation: int, x: int) -> Optional[int]:
        cells0 = self._cells(piece, rotation, x, 0)
        # Reject if any cell is already out of horizontal bounds at y=0 template
        if any(cx < 0 or cx >= self.width for cx, _ in cells0):
            return None
        y = 0
        last = None
        while True:
            cells = self._cells(piece, rotation, x, y)
            if not self._valid(grid, cells):
                break
            last = y
            y += 1
        return last

    def _apply(self, grid: np.ndarray, cells: list[tuple[int, int]]) -> tuple[np.ndarray, int]:
        g = grid.copy()
        for x, y in cells:
            g[y, x] = 1
        # clear full lines
        keep = [row for row in g if not row.all()]
        cleared = self.height - len(keep)
        if cleared:
            new = np.zeros_like(g)
            new[cleared:] = np.stack(keep, axis=0)
            g = new
        return g, cleared

    @staticmethod
    def heuristic_score(lines_cleared: int, holes: int, bumpiness: int, height: int, aggregate_height: int) -> float:
        # Higher is better. Tuned for shortlist ranking, not claimed optimal play.
        return (
            100.0 * lines_cleared
            - 40.0 * holes
            - 5.0 * bumpiness
            - 3.0 * height
            - 0.5 * aggregate_height
        )

    def enumerate_placements(self, state: Optional[GameState] = None) -> list[Placement]:
        st = state or self.state
        out: list[Placement] = []
        piece = st.current
        for rot in range(4):
            # Unique shapes for O / some rotations
            for x in range(-2, self.width + 2):
                y = self._drop_y(st.grid, piece, rot, x)
                if y is None:
                    continue
                cells = self._cells(piece, rot, x, y)
                if not self._valid(st.grid, cells):
                    continue
                new_grid, cleared = self._apply(st.grid, cells)
                feat = board_features(new_grid)
                score = self.heuristic_score(
                    cleared, feat["holes"], feat["bumpiness"], feat["height"], feat["aggregate_height"]
                )
                out.append(
                    Placement(
                        piece=piece,
                        rotation=rot,
                        x=x,
                        y=y,
                        cells=tuple(cells),
                        lines_cleared=cleared,
                        holes=feat["holes"],
                        bumpiness=feat["bumpiness"],
                        height=feat["height"],
                        aggregate_height=feat["aggregate_height"],
                        heuristic=score,
                    )
                )
        # Deduplicate identical landing cell sets
        uniq: dict[tuple[tuple[int, int], ...], Placement] = {}
        for p in out:
            key = tuple(sorted(p.cells))
            prev = uniq.get(key)
            if prev is None or p.heuristic > prev.heuristic:
                uniq[key] = p
        return sorted(uniq.values(), key=lambda p: p.heuristic, reverse=True)

    def shortlist(self, k: int, state: Optional[GameState] = None) -> list[Placement]:
        placements = self.enumerate_placements(state)
        top = placements[:k]
        letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        labelled = []
        for i, p in enumerate(top):
            labelled.append(
                Placement(
                    piece=p.piece,
                    rotation=p.rotation,
                    x=p.x,
                    y=p.y,
                    cells=p.cells,
                    lines_cleared=p.lines_cleared,
                    holes=p.holes,
                    bumpiness=p.bumpiness,
                    height=p.height,
                    aggregate_height=p.aggregate_height,
                    heuristic=p.heuristic,
                    label=letters[i],
                )
            )
        return labelled

    def state_text(self, shortlist: list[Placement], state: Optional[GameState] = None) -> str:
        st = state or self.state
        feat = board_features(st.grid)
        nxt = "".join(st.queue[:3])
        lines = [
            f"Tetris. Piece {st.current}, next {nxt or '-'}, hold none.",
            f"Stack height {feat['height']}, holes {feat['holes']}, bumpiness {feat['bumpiness']}.",
            "Choose the best placement among the options. Prefer clearing lines, fewer holes, lower height, smoother surface.",
            "Options:",
        ]
        for p in shortlist:
            lines.append(f"{p.label}: {p.option_text()}")
        return "\n".join(lines)

    def apply_placement(self, placement: Placement) -> None:
        if not self.state.alive:
            return
        cells = list(placement.cells)
        if not self._valid(self.state.grid, cells):
            self.state.alive = False
            return
        self.state.grid, cleared = self._apply(self.state.grid, cells)
        self.state.lines += cleared
        self.state.pieces += 1
        self.state.current = self._pop_next()
        if not self._spawn_ok(self.state.current):
            self.state.alive = False


# Quick sanity: one seed, enumerate a shortlist
_eng = TetrisEngine(seed=1)
_sl = _eng.shortlist(8)
print(_eng.state_text(_sl))
print("placements available:", len(_eng.enumerate_placements()), "shortlist:", len(_sl))


## 5 — Decision adapters

Each adapter returns `(choice_index, meta)` where `choice_index` indexes the shared shortlist.


In [ ]:
import json
import traceback
from abc import ABC, abstractmethod


@dataclass
class DecisionResult:
    index: int
    label: str
    latency_ms: float
    confidence: Optional[float] = None
    p_none: Optional[float] = None
    raw_choice: Any = None
    error: Optional[str] = None
    abstained: bool = False


class DecisionModel(ABC):
    name: str

    @abstractmethod
    def decide(self, state_text: str, shortlist: list[Placement]) -> DecisionResult:
        ...


class GreedyModel(DecisionModel):
    name = "greedy"

    def decide(self, state_text: str, shortlist: list[Placement]) -> DecisionResult:
        t0 = time.perf_counter()
        return DecisionResult(index=0, label=shortlist[0].label, latency_ms=(time.perf_counter() - t0) * 1000, confidence=1.0, raw_choice="rank1")


class RandomModel(DecisionModel):
    name = "random"

    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)

    def decide(self, state_text: str, shortlist: list[Placement]) -> DecisionResult:
        t0 = time.perf_counter()
        i = self.rng.randrange(len(shortlist))
        return DecisionResult(index=i, label=shortlist[i].label, latency_ms=(time.perf_counter() - t0) * 1000, confidence=1.0 / len(shortlist), raw_choice=shortlist[i].label)


class ZeroPointModel(DecisionModel):
    name = "zeropoint"

    def __init__(self, threads: int = 2):
        from zeropoint import ZeroPoint
        self.zp = ZeroPoint(threads=threads)

    def decide(self, state_text: str, shortlist: list[Placement]) -> DecisionResult:
        # Include the letter so option strings stay unique (ZeroPoint rejects duplicates).
        options = [f"{p.label}: {p.option_text()}" for p in shortlist]
        # Context without the option table — options are passed separately (ZeroPoint API).
        header = "\n".join(state_text.split("\n")[:3])
        t0 = time.perf_counter()
        try:
            d = self.zp.decide(header, options)
            latency = (time.perf_counter() - t0) * 1000
            # probabilities is a dict[str, float] keyed by option text
            if d.choice is None:
                i = int(max(range(len(options)), key=lambda j: d.probabilities.get(options[j], 0.0)))
                return DecisionResult(
                    index=i,
                    label=shortlist[i].label,
                    latency_ms=latency,
                    confidence=float(d.confidence),
                    p_none=float(d.p_none),
                    raw_choice=None,
                    abstained=True,
                )
            chosen = d.choice
            try:
                i = options.index(chosen)
            except ValueError:
                i = 0
            return DecisionResult(
                index=i,
                label=shortlist[i].label,
                latency_ms=latency,
                confidence=float(d.confidence),
                p_none=float(d.p_none),
                raw_choice=chosen,
            )
        except Exception as e:
            return DecisionResult(index=0, label=shortlist[0].label, latency_ms=(time.perf_counter() - t0) * 1000, error=f"{type(e).__name__}: {e}")


class LayaModel(DecisionModel):
    name = "laya"

    def __init__(self, model: str = "english"):
        # Ensure CPU even if a GPU runtime was selected by mistake.
        os.environ["CUDA_VISIBLE_DEVICES"] = ""
        import torch
        if torch.cuda.is_available():
            print("WARNING: torch still sees CUDA; forcing CPU tensors for Laya.")
        from laya import Router
        self.router = Router()
        self.model = model
        self.device_note = "cpu"

    def decide(self, state_text: str, shortlist: list[Placement]) -> DecisionResult:
        criteria = {p.label: p.option_text() for p in shortlist}
        questions = {
            "placement": {
                "type": "choice",
                "instructions": (
                    "Which Tetris placement is best? Prefer more line clears, fewer holes, "
                    "lower stack height, and lower bumpiness."
                ),
                "criteria": criteria,
            }
        }
        t0 = time.perf_counter()
        try:
            # Prefer explicit CPU if the Router/Agent API accepts device kwargs; fall back gracefully.
            try:
                result = self.router.predict(state_text, questions, model=self.model, device="cpu")
            except TypeError:
                result = self.router.predict(state_text, questions, model=self.model)
            latency = (time.perf_counter() - t0) * 1000
            ans = result["answers"]["placement"]
            choice = ans.get("choice")
            conf = ans.get("probability") or ans.get("confidence")
            if choice is None or choice not in criteria:
                # abstain / unbound — fall back to rank-1 but mark abstained
                return DecisionResult(
                    index=0,
                    label=shortlist[0].label,
                    latency_ms=latency,
                    confidence=float(conf) if conf is not None else None,
                    raw_choice=choice,
                    abstained=True,
                )
            i = next(j for j, p in enumerate(shortlist) if p.label == choice)
            return DecisionResult(
                index=i,
                label=choice,
                latency_ms=latency,
                confidence=float(conf) if conf is not None else None,
                raw_choice=choice,
            )
        except Exception as e:
            return DecisionResult(
                index=0,
                label=shortlist[0].label,
                latency_ms=(time.perf_counter() - t0) * 1000,
                error=f"{type(e).__name__}: {e}",
            )


class OpenAIDecisionsModel(DecisionModel):
    name = "openai_decisions"

    def __init__(self, api_key: str, model: str = "gpt-6-luna", timeout_s: float = 60.0):
        self.api_key = api_key
        self.model = model
        self.timeout_s = timeout_s
        self._client = None
        try:
            from openai import OpenAI
            self._client = OpenAI(api_key=api_key, timeout=timeout_s)
        except Exception:
            self._client = None

    def decide(self, state_text: str, shortlist: list[Placement]) -> DecisionResult:
        choices = [p.label for p in shortlist]
        instructions = (
            "Pick the best Tetris placement letter. Prefer more line clears, fewer holes, "
            "lower stack height, and lower bumpiness. The option lines describe consequences."
        )
        t0 = time.perf_counter()
        try:
            if self._client is not None and hasattr(self._client, "decisions"):
                resp = self._client.decisions.create(
                    model=self.model,
                    input=state_text,
                    questions=[
                        {
                            "type": "choice",
                            "name": "placement",
                            "instructions": instructions,
                            "choices": choices,
                        }
                    ],
                )
                # SDK object or dict-like
                if hasattr(resp, "model_dump"):
                    data = resp.model_dump()
                elif isinstance(resp, dict):
                    data = resp
                else:
                    data = json.loads(resp.json()) if hasattr(resp, "json") else {"answers": []}
            else:
                import requests
                r = requests.post(
                    "https://api.openai.com/v1/decisions",
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "Content-Type": "application/json",
                    },
                    json={
                        "model": self.model,
                        "input": state_text,
                        "questions": [
                            {
                                "type": "choice",
                                "name": "placement",
                                "instructions": instructions,
                                "choices": choices,
                            }
                        ],
                    },
                    timeout=self.timeout_s,
                )
                r.raise_for_status()
                data = r.json()

            latency = (time.perf_counter() - t0) * 1000
            answers = data.get("answers") or []
            ans = answers[0] if answers else {}
            if ans.get("type") == "refusal":
                return DecisionResult(index=0, label=shortlist[0].label, latency_ms=latency, abstained=True, raw_choice="refusal")
            choice = ans.get("choice")
            conf = ans.get("confidence")
            if choice not in choices:
                return DecisionResult(index=0, label=shortlist[0].label, latency_ms=latency, confidence=conf, raw_choice=choice, abstained=True)
            i = choices.index(choice)
            return DecisionResult(index=i, label=choice, latency_ms=latency, confidence=float(conf) if conf is not None else None, raw_choice=choice)
        except Exception as e:
            return DecisionResult(
                index=0,
                label=shortlist[0].label,
                latency_ms=(time.perf_counter() - t0) * 1000,
                error=f"{type(e).__name__}: {e}",
            )


def build_models(cfg: Config) -> dict[str, DecisionModel]:
    models: dict[str, DecisionModel] = {}
    if cfg.run_greedy:
        models["greedy"] = GreedyModel()
    if cfg.run_random:
        models["random"] = RandomModel(seed=0)
    if cfg.run_zeropoint:
        print("Loading ZeroPoint…")
        models["zeropoint"] = ZeroPointModel(threads=cfg.zeropoint_threads)
        print("ZeroPoint ready")
    if cfg.run_laya:
        print("Loading Laya on CPU (first download can take several minutes)…")
        models["laya"] = LayaModel(model=cfg.laya_model)
        print("Laya ready")
    if cfg.run_openai and OPENAI_API_KEY:
        print("OpenAI Decisions client ready")
        models["openai_decisions"] = OpenAIDecisionsModel(
            api_key=OPENAI_API_KEY, model=cfg.openai_model, timeout_s=cfg.openai_timeout_s
        )
    print("models:", list(models))
    return models


MODELS = build_models(CFG)


## 6 — Game runner and metrics


In [ ]:
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm


def play_game(model: DecisionModel, seed: int, max_moves: int, shortlist_k: int, width: int, height: int) -> dict:
    engine = TetrisEngine(width=width, height=height, seed=seed)
    latencies = []
    errors = 0
    abstentions = 0
    agreement_with_greedy = 0
    moves = 0

    while engine.state.alive and moves < max_moves:
        sl = engine.shortlist(shortlist_k)
        if not sl:
            engine.state.alive = False
            break
        state_text = engine.state_text(sl)
        decision = model.decide(state_text, sl)
        latencies.append(decision.latency_ms)
        if decision.error:
            errors += 1
        if decision.abstained:
            abstentions += 1
        if decision.index == 0:
            agreement_with_greedy += 1
        idx = decision.index if 0 <= decision.index < len(sl) else 0
        engine.apply_placement(sl[idx])
        moves += 1

    arr = np.array(latencies, dtype=float) if latencies else np.array([0.0])
    return {
        "model": model.name,
        "seed": seed,
        "lines": engine.state.lines,
        "pieces": engine.state.pieces,
        "survived": bool(engine.state.alive and engine.state.pieces >= max_moves),
        "topped_out": not engine.state.alive,
        "moves": moves,
        "errors": errors,
        "abstentions": abstentions,
        "greedy_agreement": agreement_with_greedy / max(moves, 1),
        "latency_ms_mean": float(arr.mean()),
        "latency_ms_p50": float(np.percentile(arr, 50)),
        "latency_ms_p95": float(np.percentile(arr, 95)),
        "latency_ms_sum": float(arr.sum()),
    }


def run_experiment(models: dict[str, DecisionModel], seeds: tuple[int, ...], max_moves: int, cfg: Config) -> pd.DataFrame:
    rows = []
    work = [(name, seed) for name in models for seed in seeds]
    for name, seed in tqdm(work, desc="games"):
        rows.append(
            play_game(
                models[name],
                seed=seed,
                max_moves=max_moves,
                shortlist_k=cfg.shortlist_k,
                width=cfg.board_width,
                height=cfg.board_height,
            )
        )
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("model", sort=False)
    out = g.agg(
        games=("seed", "count"),
        lines_mean=("lines", "mean"),
        lines_std=("lines", "std"),
        survival_rate=("survived", "mean"),
        pieces_mean=("pieces", "mean"),
        error_games=("errors", lambda s: int((s > 0).sum())),
        abstentions_sum=("abstentions", "sum"),
        moves_sum=("moves", "sum"),
        greedy_agreement=("greedy_agreement", "mean"),
        latency_p50_ms=("latency_ms_p50", "mean"),
        latency_p95_ms=("latency_ms_p95", "mean"),
        latency_mean_ms=("latency_ms_mean", "mean"),
    ).reset_index()
    out["abstention_rate"] = out["abstentions_sum"] / out["moves_sum"].clip(lower=1)
    return out.drop(columns=["abstentions_sum", "moves_sum"])


## 7 — Smoke test (recommended before the full run)

1 seed × 15 moves × enabled models. Verifies installs and API access.


In [ ]:
smoke_df = run_experiment(MODELS, seeds=CFG.smoke_seeds, max_moves=CFG.smoke_max_moves, cfg=CFG)
display(smoke_df)
display(summarize(smoke_df))
smoke_df.to_csv(f"{CFG.results_dir}/smoke_games.csv", index=False)
print("smoke OK — inspect latencies and errors before launching the full run")


## 8 — Full experiment

Default: 10 seeds × 100 moves. On Colab **CPU**, Laya dominates wall time. You can shrink `CFG.seeds` / `CFG.max_moves` above and re-run model build + this section.


In [ ]:
full_df = run_experiment(MODELS, seeds=CFG.seeds, max_moves=CFG.max_moves, cfg=CFG)
summary_df = summarize(full_df)

full_path = f"{CFG.results_dir}/games.csv"
summary_path = f"{CFG.results_dir}/summary.csv"
full_df.to_csv(full_path, index=False)
summary_df.to_csv(summary_path, index=False)

display(summary_df)
display(full_df.head(20))
print("wrote", full_path, "and", summary_path)


## 9 — Plots


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Lines per game
order = list(summary_df["model"])
ax = axes[0]
for model in order:
    vals = full_df.loc[full_df["model"] == model, "lines"]
    ax.scatter([model] * len(vals), vals, alpha=0.7)
    ax.scatter([model], [vals.mean()], marker="_", s=400, color="black")
ax.set_title("Lines / game")
ax.tick_params(axis="x", rotation=30)
ax.set_ylabel("lines")

# Latency p50
ax = axes[1]
ax.bar(summary_df["model"], summary_df["latency_p50_ms"])
ax.set_title("Decision latency p50 (ms)")
ax.tick_params(axis="x", rotation=30)
ax.set_ylabel("ms")

# Survival
ax = axes[2]
ax.bar(summary_df["model"], summary_df["survival_rate"])
ax.set_ylim(0, 1.05)
ax.set_title(f"Survival rate (≥ {CFG.max_moves} moves)")
ax.tick_params(axis="x", rotation=30)

fig.suptitle("ZeroPoint Tetris Decision Bench (CPU-first)", y=1.02)
fig.tight_layout()
plot_path = f"{CFG.results_dir}/summary_plots.png"
fig.savefig(plot_path, dpi=140, bbox_inches="tight")
print("wrote", plot_path)
plt.show()


## 10 — Developer notes / interpretation

### Reading the table

- **`greedy`**: always picks the top heuristic option. Upper bound *given this shortlist*. If a model beats greedy on lines, either luck on ties or the heuristic rank order is imperfect.
- **`random`**: floor. If a model is near random, the shortlist text is not being used well.
- **`zeropoint` vs `laya`**: fair local CPU comparison. Compare `latency_p50_ms`, lines, survival, and footprint (ZeroPoint ships at ~89 MB; Laya’s English checkpoint is hundreds of MB–GB with torch).
- **`openai_decisions`**: cloud baseline. Expect higher latency variance and API cost; not a CPU peer.

### Footprint (approximate, for the write-up)

| Model | Approx. local footprint | Network |
|-------|-------------------------|---------|
| ZeroPoint | ~89 MB wheel + onnxruntime | none |
| Laya | torch + ~400M-param checkpoint | Hub download once |
| OpenAI Decisions | client only | every decision |

### Reproducibility checklist

- [ ] `CUDA_VISIBLE_DEVICES=""` set before importing torch/laya
- [ ] Same `seeds`, `max_moves`, `shortlist_k`
- [ ] Zero-shot Laya (no Tetris fine-tune)
- [ ] Pinned `zeropoint==1.1.0`
- [ ] Saved `results/games.csv` + `results/summary.csv`

### Known limitations

1. Drop-only placements (no holds, kicks, or combos) — fine for a decision-model demo, not a tournament bot.
2. Option wording is English feature strings; models that prefer letters-only vs feature text may differ — both receive the same strings.
3. When a model abstains / returns `none`, this harness falls back to rank-1 so the game continues; abstentions are counted separately.
4. OpenAI pricing/latency depend on region and load; treat cloud numbers as indicative.

### Next steps (outside this notebook)

- Record a short video of a browser UI using the same shortlist protocol.
- Add a manufacturing ECR-triage notebook with the same adapter pattern.
- Decide whether ZeroPoint needs new “connector” helpers (batching, logging, threshold helpers) based on friction you hit here.


## 11 — Package versions (paste into issues / PRs)


In [ ]:
import importlib.metadata as md

for pkg in ["zeropoint", "laya", "openai", "numpy", "onnxruntime", "torch", "tokenizers"]:
    try:
        print(f"{pkg:15} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:15} (not installed)")
